In [1]:
# %% [0] Phase 1 -- layered-medium Green's function for the TFLN stack
#
# Builds and validates the mixed-potential Green's function that is the kernel
# of the layered-medium MoM (Route C).  Same-interface scalar-potential G_q and
# vector-potential G_A on the metal plane, inverted from the transmission-line
# spectral kernels by a Sommerfeld integral with quasi-static singularity
# extraction and a Hankel-split deformed contour (real head past the surface-
# wave poles, then two exponentially-decaying rays).
#
# Deliverables written by this notebook:
#   green_function_table.npz   the validated 1D table G_q(rho), G_A(rho)
#   te0_pole.json              the TE0 (and TM0) surface-wave pole
#   phase1_greens.png          the 3-panel figure
#   phase1_report.md           the one-page report
#
# The Green's function machinery lives in layered_greens.py (reused by later
# phases); the stack constants live in stack_params.py.
import json, time, warnings
import numpy as np
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from scipy import special, integrate
from layered_greens import Stack, C0, EPS0, MU0
import stack_params as sp

f = sp.F0
k0 = 2 * np.pi * f / C0
print(f"table stack: air | LN {sp.SLAB_REF*1e6:.3f} um etched slab "
      f"(film {sp.TFLN*1e6:.2f} um - median etch {sp.ETCH_MEDIAN*1e6:.3f}; eps {sp.EPS_LN}) | "
      f"SiO2 {sp.BOX_H*1e6:.1f} um (eps {sp.EPS_SIO2}) | "
      f"Si {sp.SI_H*1e6:.0f} um (eps {sp.EPS_SI}) | air   f = {f/1e9:.0f} GHz")


table stack: air | LN 0.225 um etched slab (film 0.46 um - median etch 0.235; eps 34.7) | SiO2 4.7 um (eps 3.9) | Si 550 um (eps 11.7) | air   f = 60 GHz


In [2]:
# %% [1] Build and save the Green's function table (device stack, etched LN slab)
# ============================================================
# The RF electrodes sit on the etched LiNbO3 slab, not the full film; the table
# is built at the dataset-median etched thickness SLAB_REF.  device_layers(t_LN)
# is parametrised so Phase 2 can rebuild/interpolate the kernel at each geometry's
# t_LN = TFLN - ETCH_DEPTH (see V1.7 for why this matters for the near field).
DEV = Stack(sp.EPS_AIR, sp.device_layers(t_LN=sp.SLAB_REF, lossy=True), sp.EPS_AIR, f=f)
rho = np.logspace(np.log10(1e-8), np.log10(1e-2), 90)     # 10 nm ... 10 mm
t0 = time.time()
G_A, G_q = DEV.table(rho)
t_table = time.time() - t0
np.savez("green_function_table.npz", rho=rho, G_A=G_A, G_q=G_q, f=f,
         t_LN=sp.SLAB_REF,
         layers=np.array([(e.real, e.imag, d) for e, d in sp.device_layers(t_LN=sp.SLAB_REF)]),
         top_eps=sp.EPS_AIR, bot_eps=sp.EPS_AIR)


In [3]:
# %% [2] Surface-wave poles: zeros of the kernel denominator (eta_up+eta_dn)
# ============================================================
# A bound surface wave is a pole of the SAME-INTERFACE kernel (transverse
# resonance), found directly from the denominator the Sommerfeld integrator
# evaluates.  The V1.4 target 2.5414 is the UNETCHED-film TE0, so the pole finder
# is validated on the unetched stack; the etched table-stack poles (which set the
# contour breakpoints) are recorded separately.
UNETCH = Stack(sp.EPS_AIR, sp.device_layers(t_LN=sp.TFLN, lossy=False), sp.EPS_AIR, f=f)
n_te0 = UNETCH.surface_waves("TE")[0]
n_tm0 = UNETCH.surface_waves("TM")[0]
resid = abs(UNETCH._sw_den(n_te0, "TE"))
DEV_LL = Stack(sp.EPS_AIR, sp.device_layers(t_LN=sp.SLAB_REF, lossy=False), sp.EPS_AIR, f=f)
te_tab = DEV_LL.surface_waves("TE")[0]
tm_tab = DEV_LL.surface_waves("TM")[0]
json.dump({"n_TE0_unetched": n_te0, "n_TM0_unetched": n_tm0, "residual": resid,
           "n_TE0_table": te_tab, "n_TM0_table": tm_tab, "t_LN_table": sp.SLAB_REF},
          open("te0_pole.json", "w"), indent=2)


In [4]:
# %% [3] Validation gate  V1.1 - V1.6 (pass/fail) + V1.7 (informational)
# ============================================================
res = {}

# V1.1 free space ---------------------------------------------------------
S = Stack(1.0, [], 1.0, f=f)
e11 = max(abs(S.Gq(r) - np.exp(-1j*k0*r)/(4*np.pi*EPS0*r)) /
          abs(np.exp(-1j*k0*r)/(4*np.pi*EPS0*r)) for r in (1e-6, 1e-5, 1e-4))
res["V1.1"] = ("free-space limit", e11, 1e-4, e11 < 1e-4)

# V1.2 PEC image (vacuum layer h over PEC; image charge -q at depth 2h) ----
h = 5e-6
SP = Stack(1.0, [(1.0, h)], 1e7, f=f)
def _img(r):
    R = np.sqrt(r**2 + (2*h)**2)
    return (np.exp(-1j*k0*r)/r - np.exp(-1j*k0*R)/R) / (4*np.pi*EPS0)
e12 = max(abs(SP.Gq(r) - _img(r)) / abs(_img(r)) for r in (1e-6, 5e-6, 2e-5))
res["V1.2"] = ("PEC-ground image", e12, 1e-3, e12 < 1e-3)

# V1.3 single dielectric interface eps2=3.9 (static two-dielectric image) --
S2 = Stack(1.0, [], 3.9, f=f)
e13 = max(abs(S2.Gq(r).real - 1/(2*np.pi*EPS0*(1+3.9)*r)) /
          (1/(2*np.pi*EPS0*(1+3.9)*r)) for r in (1e-7, 1e-6, 1e-5))
res["V1.3"] = ("single interface eps=3.9", e13, 1e-2, e13 < 1e-2)

# V1.4 TE0 pole of the UNETCHED film vs the validated reference ------------
e14 = abs(n_te0 - sp.N_TE0_REF)
res["V1.4"] = ("TE0 pole, unetched (=2.5414)", e14, 1e-3, e14 < 1e-3)

# V1.5 independent-integrator cross-check ---------------------------------
# Production G_q uses a Hankel-split deformed contour; re-evaluate it on the
# DEVICE stack with a completely different tail method (real-axis partition at
# J0 zeros + Mosig weighted-averages acceleration).  Agreement to ~1e-6
# validates the production Sommerfeld integrator against an independent scheme.
def gq_indep(stack, rho_):
    C = stack._asym("TM"); spec = stack.Gq_spectral
    rem = lambda kr: (spec(kr) - C/kr) * special.j0(kr*rho_) * kr / (2*np.pi)
    a = 6.0 * stack.n_max * k0
    bps = sorted(set([k0*np.sqrt(abs(e)) for e in [stack.top_eps, stack.bot_eps]
                      + [e for e, _ in stack.layers]] + [n*k0 for n in stack._poles()]))
    seg = [0.0] + [b for b in bps if 0 < b < a] + [a]
    head = sum(integrate.quad(lambda t: rem(t).real, u, v, limit=200)[0]
               + 1j*integrate.quad(lambda t: rem(t).imag, u, v, limit=200)[0]
               for u, v in zip(seg[:-1], seg[1:]))
    z = special.jn_zeros(0, 40) / rho_
    brk = [a] + [q for q in z if q > a]
    parts = [integrate.quad(lambda t: rem(t).real, brk[i], brk[i+1], limit=100)[0]
             + 1j*integrate.quad(lambda t: rem(t).imag, brk[i], brk[i+1], limit=100)[0]
             for i in range(len(brk)-1)]
    A = list(np.cumsum(parts))
    while len(A) > 1:                                  # repeated averaging
        A = [(A[i] + A[i+1]) / 2 for i in range(len(A)-1)]
    return head + A[0] + C/(2*np.pi*rho_)
e15 = max(abs(DEV.Gq(r) - gq_indep(DEV, r)) / abs(DEV.Gq(r)) for r in (3e-6, 1e-5, 3e-5))
res["V1.5"] = ("independent-integrator xcheck", e15, 1e-6, e15 < 1e-6)

# V1.6 table interpolation self-consistency (numerical, not physics) ------
from scipy.interpolate import CubicSpline
lr = np.log(rho)
csr = CubicSpline(lr, G_q.real); csi = CubicSpline(lr, G_q.imag)
rho_mid = np.sqrt(rho[20:70:8] * rho[21:71:8])          # intermediate field, off-grid
def direct(rm):
    return DEV._sommerfeld(rm, DEV.Gq_spectral, DEV._asym("TM"))
e16 = max(abs((csr(np.log(rm)) + 1j*csi(np.log(rm))) - direct(rm)) / abs(direct(rm))
          for rm in rho_mid)
res["V1.6"] = ("table interp (self-consist.)", e16, 1e-3, e16 < 1e-3)

# V1.7 (informational, no pass/fail) LN-thickness sensitivity -------------
# G at the etched-slab extremes t_LN = 0.10 and 0.36 um.  G_A is insensitive;
# G_q is sensitive at short range -> Phase 2 must use the per-geometry t_LN for
# the near-field (capacitive) terms.
Slo = Stack(sp.EPS_AIR, sp.device_layers(t_LN=0.10e-6, lossy=True), sp.EPS_AIR, f=f)
Shi = Stack(sp.EPS_AIR, sp.device_layers(t_LN=0.36e-6, lossy=True), sp.EPS_AIR, f=f)
sens = {r: (abs(Shi.Gq(r) - Slo.Gq(r)) / abs(Slo.Gq(r)),
            abs(Shi.GA(r) - Slo.GA(r)) / abs(Slo.GA(r))) for r in (1e-6, 1e-5, 1e-4)}

# ---- report to stdout ---------------------------------------------------
print("="*70)
print("PHASE 1 VALIDATION GATE")
print("-"*70)
allpass = True
for k in ["V1.1", "V1.2", "V1.3", "V1.4", "V1.5", "V1.6"]:
    name, val, thr, ok = res[k]
    allpass &= ok
    print(f"  {k}  {name:<30} {val:.2e}  (< {thr:.0e})  {'PASS' if ok else 'FAIL'}")
print("-"*70)
print("  V1.7  LN-thickness sensitivity (t_LN 0.10 vs 0.36 um), INFORMATIONAL:")
for r in (1e-6, 1e-5, 1e-4):
    dq, da = sens[r]
    print(f"          rho={r*1e6:6.1f} um   |dGq|/|Gq|={dq:.2e}   |dGA|/|GA|={da:.2e}")
print("-"*70)
print(f"  poles: TE0 unetched {n_te0:.5f} (ref 2.5414, resid {resid:.1e}), "
      f"TM0 unetched {n_tm0:.5f}")
print(f"         table stack (t_LN={sp.SLAB_REF*1e6:.3f} um): "
      f"TE0 {te_tab:.5f}, TM0 {tm_tab:.5f}")
print(f"  table: {len(rho)} rho points built in {t_table:.1f} s")
print(f"  {'ALL PASS' if allpass else 'SOME FAILED'}  (V1.1-V1.4 physics, "
      f"V1.5 independent numerics, V1.6 interpolation self-consistency)")
print("="*70)


PHASE 1 VALIDATION GATE
----------------------------------------------------------------------
  V1.1  free-space limit               8.41e-14  (< 1e-04)  PASS
  V1.2  PEC-ground image               8.66e-08  (< 1e-03)  PASS
  V1.3  single interface eps=3.9       2.62e-04  (< 1e-02)  PASS
  V1.4  TE0 pole, unetched (=2.5414)   1.00e-05  (< 1e-03)  PASS
  V1.5  independent-integrator xcheck  1.27e-13  (< 1e-06)  PASS
  V1.6  table interp (self-consist.)   4.72e-05  (< 1e-03)  PASS
----------------------------------------------------------------------
  V1.7  LN-thickness sensitivity (t_LN 0.10 vs 0.36 um), INFORMATIONAL:
          rho=   1.0 um   |dGq|/|Gq|=3.63e-01   |dGA|/|GA|=3.15e-05
          rho=  10.0 um   |dGq|/|Gq|=3.74e-02   |dGA|/|GA|=1.98e-04
          rho= 100.0 um   |dGq|/|Gq|=7.42e-03   |dGA|/|GA|=8.46e-04
----------------------------------------------------------------------
  poles: TE0 unetched 2.54141 (ref 2.5414, resid 6.4e-05), TM0 unetched 3.31373
         table st

In [5]:
# %% [4] Figure and report
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].loglog(rho*1e6, np.abs(G_q), color="#2c7fb8")
ax[0].set_xlabel(r"$\rho$ ($\mu$m)"); ax[0].set_ylabel(r"$|G_q|$ (V m / C)")
ax[0].set_title(r"(a) scalar-potential kernel $G_q(\rho)$")
ax[1].loglog(rho*1e6, np.abs(G_A), color="#e67e22")
ax[1].set_xlabel(r"$\rho$ ($\mu$m)"); ax[1].set_ylabel(r"$|G_A|$ (H/m)")
ax[1].set_title(r"(b) vector-potential kernel $G_A(\rho)$")
# (c) lossy spectral kernels vs effective index, full range, both poles peak
kr = np.linspace(1.001*k0, (DEV.n_max - 0.02)*k0, 6000)
gt = np.abs([DEV.Gq_spectral(k) for k in kr])
ht = np.abs([DEV.GA_spectral(k) for k in kr])
ax[2].semilogy(kr/k0, gt/gt.max(), color="#2c7fb8", label=r"$|\tilde G_q|$ (TM)")
ax[2].semilogy(kr/k0, ht/ht.max(), color="#e67e22", label=r"$|\tilde G_A|$ (TE)")
ax[2].axvline(tm_tab, color="#2c7fb8", ls=":", lw=1)
ax[2].axvline(te_tab, color="#e67e22", ls=":", lw=1)
ax[2].set_xlabel(r"$k_\rho / k_0$  (= effective index)")
ax[2].set_ylabel("normalised spectral kernel (lossy)")
ax[2].set_title(f"(c) surface-wave poles: TE {te_tab:.3f}, TM {tm_tab:.3f}")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("phase1_greens.png", dpi=130); plt.close()

# ============================================================
# report
# ============================================================
with open("phase1_report.md", "w") as fh:
    fh.write("# Phase 1 report -- layered-medium Green's function\n\n")
    fh.write("Method: a surface-integral-equation kernel with unknowns only on "
             "the metal surface (Phase 2); the metal's finite conductivity enters "
             "as a surface-impedance BC, and the layered substrate is folded "
             "*exactly* into the Green's function -- nothing is averaged or "
             "treated volumetrically.\n\n")
    fh.write(f"Table stack: air | LiNbO3 {sp.SLAB_REF*1e6:.3f} um etched slab "
             f"(= {sp.TFLN*1e6:.2f} um film - {sp.ETCH_MEDIAN*1e6:.3f} um median "
             f"etch; eps {sp.EPS_LN}) | SiO2 {sp.BOX_H*1e6:.1f} um (eps {sp.EPS_SIO2}) "
             f"| Si {sp.SI_H*1e6:.0f} um (eps {sp.EPS_SI}, sigma {sp.SIGMA_SI:g}) | air, "
             f"at {f/1e9:.0f} GHz.  device_layers(t_LN) is parametrised: the etched "
             f"thickness t_LN = 0.46 um - ETCH_DEPTH varies across the dataset "
             f"([0.10, 0.36] um), and CST's own HF Multilayer background uses this "
             f"etched SLAB_H, not the full film.\n\n")
    fh.write(f"**TE0 surface-wave pole (unetched film): n = {n_te0:.5f}** "
             f"(ref 2.5414, residual {resid:.1e}); TM0 (unetched) n = {n_tm0:.5f}.  "
             f"Table stack (t_LN = {sp.SLAB_REF*1e6:.3f} um): "
             f"TE0 {te_tab:.5f}, TM0 {tm_tab:.5f}.\n\n")
    fh.write("## Validation gate\n\n| test | what | error | threshold | result |\n")
    fh.write("|---|---|---|---|---|\n")
    for k in ["V1.1", "V1.2", "V1.3", "V1.4", "V1.5", "V1.6"]:
        name, val, thr, ok = res[k]
        fh.write(f"| {k} | {name} | {val:.2e} | {thr:.0e} | "
                 f"{'PASS' if ok else 'FAIL'} |\n")
    fh.write(f"\nV1.1-V1.6 {'all PASS' if allpass else 'did NOT all pass'}. "
             f"Table: {len(rho)} log-spaced rho points (10 nm - 10 mm), "
             f"built in {t_table:.1f} s.\n\n")
    fh.write("**What each gate proves (honest).** V1.1-V1.4 are the independent "
             "physics checks (free-space Sommerfeld identity, PEC image, static "
             "two-dielectric image, and the known unetched TE0 surface wave). "
             "V1.5 is a numerical cross-check of the production integrator against "
             "an independent tail-summation method (not a new physics limit). "
             "V1.6 is interpolation self-consistency of the saved table -- it "
             "confirms the sampling, not the physics.\n\n")
    fh.write("## V1.7 -- LN-thickness sensitivity (informational)\n\n")
    fh.write("G at the etched-slab extremes t_LN = 0.10 vs 0.36 um:\n\n")
    fh.write("| rho | |dGq|/|Gq| | |dGA|/|GA| |\n|---|---|---|\n")
    for r in (1e-6, 1e-5, 1e-4):
        dq, da = sens[r]
        fh.write(f"| {r*1e6:.0f} um | {dq:.2e} | {da:.2e} |\n")
    fh.write("\nG_A is insensitive to the LN slab thickness (< 0.1 %); G_q is "
             "strongly sensitive at short range (~36 % at 1 um, ~4 % at 10 um, "
             "< 1 % at 100 um).  Consequence for Phase 2: the near-field "
             "(capacitive) MoM terms must use the per-geometry t_LN = 0.46 um - "
             "ETCH_DEPTH (build or interpolate the kernel per row); the far-field "
             "and all of G_A are effectively thickness-independent.  Note the "
             "*integrated* line parameters are only weakly sensitive to ETCH_DEPTH "
             "(empirical |r| < 0.07 with n_m, z0 over the 500 rows) because they "
             "are dominated by the electrode geometry -- but that downstream "
             "cancellation is not a licence to build the kernel at the wrong t_LN.\n\n")
    fh.write("## Known approximation carried into Phase 2 (validation risk)\n\n")
    fh.write("LiNbO3 is modelled isotropically with the geometric-mean proxy "
             "eps = sqrt(28*43) ~ 34.7.  This is NOT pinned by the TE0 pole (that "
             "pole is Si-slab dominated and shifts only 2.540 -> 2.542 as eps_LN "
             "goes 28 -> 43).  CST uses anisotropic LN (43, 28, 43), which changes "
             "the local permittivity under the metal by ~+/-20 %.  If Phase 2's "
             "delta_alpha misses alpha_delta_val by ~10-20 %, the anisotropic-LN "
             "kernel is the first thing to check.\n\n")
    fh.write("Figure: `phase1_greens.png` -- (a) |G_q|, (b) |G_A|, "
             "(c) lossy spectral kernels with both surface-wave poles (each a peak).\n\n")
    fh.write(f"G_q at rho=1 um: {G_q[np.argmin(abs(rho-1e-6))]:.4e};  "
             f"at rho=100 um: {G_q[np.argmin(abs(rho-1e-4))]:.4e}.\n")
print("wrote green_function_table.npz, te0_pole.json, phase1_greens.png, phase1_report.md")


wrote green_function_table.npz, te0_pole.json, phase1_greens.png, phase1_report.md
